In [15]:
import pandas as pd
import os
from uuid import uuid4
from transformers import AutoTokenizer
from math import ceil
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict
from tqdm import tqdm


tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2" ## adjust tokenization model to the one that is used in the embedding/retriever achitecture # "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp"
)

token_length = 256 # adjust to maximal token length
document_limit = 1000
dataset = 'dev' # test

# restricting legth (make room for cls token and paragraph seperators) 
TOK_LEN = token_length - 10

In [16]:
# read  data
df = pd.read_parquet(f'/raid/deallab/SF_RAG_Data/ASQA/{dataset}.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who is the original artist of sound of silence? 

qa_pairs :
[{'context': 'Sounds of Silence is the second studio album by Simon & Garfunkel, released on January 17, 1966. The album\'s title is a slight modification of the title of the duo\'s first major hit, "The Sound of Silence", which originally was released as "The Sounds of Silence". The song had earlier been released in an acoustic version on the album "Wednesday Morning, 3 A.M.", and later on the soundtrack to the movie "The Graduate". Without the knowledge of Paul Simon or Art Garfunkel, electric guitars, bass and drums were overdubbed by Columbia Records staff producer Tom Wilson on June 15, 1965. This new version was released as a single in September 1965, and opens the album.', 'question': 'Who is the original artist of sound of silence, the song, released in 1964?', 'short_answers': array(['Simon & Garfunkel', 'Paul Simon and Art Garfunkel',
        'Art Garfunkel', 'Paul Simon'], dtype=object), 'wikip

In [21]:
sample = df.loc[4]
[answ['long_answer'] for answ in sample['annotations']]

['When the Virginia state park system was formed on June 15, 1936, there were only six state parks in the entire state. As of 2016, that number had gone up to 38 state parks. ',
 'Virginia opened its entire state park system on June 15, 1936 as a six-park system. The six original state parks were Seashore State Park, now First Landing State Park, Westmoreland State Park, Staunton River State Park, Douthat State Park, Fairy Stone State Park, and Hungry Mother State Park. Natural Bridge State Park officially opened on September 24, 2016, making this 38 parks in VA. Today, the park system now oversees 43 parks.']

In [3]:
#parse tables to text
def get_table(table):
    table_text = []
    for i, tr in enumerate(table.find('tbody').findChildren("tr" , recursive=False)):
        tr_text = tr.get_text()
        tr_text = re.sub(r'\n+',';',tr_text).strip(';')
        if not tr_text: continue
        if table_text == []:
            tr_text = '\n#### Table: ' + tr_text
        table_text.append(tr_text)

    return '\n'.join(table_text)

# parse pars to text
def get_p(par):
    p_text = par.get_text()
    p_text = p_text.replace('\n','')
    return p_text

def get_h(heading):
    h = heading.find(['h1', 'h2', 'h3','h4', 'h5'])
    try:
        heading_type = int(re.search(r'<h(\d)', str(h)).group(1))
    except:
        print(heading)
        raise
    h_text ='\n' + ' '.join(['#'*heading_type,h.get_text()])
    return h_text

#pars unordered lists to text
def get_ul(ul):
    list_text = []
    for li in ul.find_all('li'):
        list_text.append('* ' + li.get_text())
    return '\n'.join(list_text)

# pars ordered list to text
def get_ol(ol):
    list_text = []
    for i, li in enumerate(ol.find_all('li')):
        if li.get_text():
            list_text.append(' '.join([str(i+1),li.get_text().replace('\n','')]))
    return '\n'.join(list_text)
        

# parse whole document
def parse_document(doc):
    content = doc.find_all(['div', 'p', 'table', 'ul', 'ol'])
    document  = []
    for cont in content:
        #stop condition
        if cont.name == 'div' and cont.find(['h1', 'h2', 'h3','h4', 'h5'], id=['See_also', 'References']):
            break
        
        #get headining
        if cont.name == 'div' and cont.has_attr('class') and  'mw-heading' in cont['class']:
            document.append(get_h(cont))
        # get par
        elif cont.name == 'p':
            par = get_p(cont)
            if par:
                document.append(par)
        #get ul
        elif cont.name == 'ul':
            document.append(get_ul(cont))
        #get ol
        elif cont.name == 'ol':
            document.append(get_ol(cont))
        #get table
        elif cont.name == 'table':
            if cont.has_attr('class') and 'metadata' in cont['class']: continue
            document.append(get_table(cont))
        # explore div
        elif cont.name == 'div':
            document.append(parse_document(cont))

    return '\n'.join(document).strip('\n')



In [ ]:
# create embedding document dataset.
evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'question', 'text'])

#creating question question pairs for retrival training (not implemented)
qq_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions'])

evidence_output_path='/raid/deallab/SF_RAG_Data/ASQA/embedding_train.csv'
follow_up_output_path='/raid/deallab/SF_RAG_Data/ASQA/follow_up_train.csv'
evidence_df.to_csv('/raid/deallab/SF_RAG_Data/ASQA/embedding_train.csv', index=False)
qq_df.to_csv(f'/raid/deallab/SF_RAG_Data/ASQA/follow_up_train.csv', index=False)

#fetched_documents = {}
for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
    try:
        if idx == document_limit: break # change number of document to chunk/process
        sample_id = row['sample_id']
        evidences = row['wikipages']
        q1 = row['ambiguous_question']
        q2 = defaultdict(list)
        follow_up_question = '\n'.join([f'### {q["question"]}' for q in row['qa_pairs']])
        for q in row['qa_pairs']:
            question = q['question']
            wikipage = q['wikipage']
            if not question or not wikipage: continue
            q2[wikipage].append(question)
        for evidence in evidences:
            url = evidence['url']
            #if url in fetched_documents: continue
            #fetched_documents[url] = []
            page = requests.get(url)
            
            # Create a BeautifulSoup object
            soup = BeautifulSoup(page.text, 'html.parser')
            # get title
            title = soup.find(id='firstHeading').get_text()
            
            #extract content
            content = soup.find(class_='mw-content-ltr')
            parsed_doc = parse_document(content)
            
            # chunk document
            documents = [[]]
            
            for par in re.split(r'(?=\n#{1,4})', parsed_doc):
                tokenized_par = tokenizer.encode(par, add_special_tokens = False)
                length = len(tokenized_par)
                if len(documents[-1]) + length < TOK_LEN:
                    documents[-1].extend(tokenized_par)
                elif length > TOK_LEN:
                    begin = 0 
                    while begin < length:
                        if begin + TOK_LEN >= length:
                            documents.append(tokenized_par[begin:])
                            break
                        documents.append(tokenized_par[begin:begin + TOK_LEN])
                        begin += TOK_LEN - int(TOK_LEN * 0.1)
                else:
                    documents.append(tokenized_par)
                
            # print(len(documents))
            for doc in documents:
                doc_text = tokenizer.decode(doc)
                id = uuid4()
                #fetched_documents[url].append(id)
                evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, q1, doc_text]
            
            if title in q2:
                for question in q2[title]:
                    for doc in documents:
                        doc_text = tokenizer.decode(doc)
                        id = uuid4()
                        #fetched_documents[url].append(id)
                        evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, question, doc_text]
        qq_df.loc[len(qq_df)] = [uuid4(), sample_id,  question, follow_up_question]
    except:
        continue
    if (idx + 1 )% 10 == 0:
        evidence_df.to_csv(evidence_output_path, mode='a', header=not os.path.exists(evidence_output_path), index=False)
        qq_df.to_csv(follow_up_output_path, mode='a', header=not os.path.exists(evidence_output_path), index=False)
        evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'question', 'text'])
        qq_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions'])
        

evidence_df.to_csv(evidence_output_path, mode='a', header=not os.path.exists(evidence_output_path), index=False)
qq_df.to_csv(follow_up_output_path, mode='a', header=not os.path.exists(evidence_output_path), index=False)

  0%|          | 0/1000 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (10448 > 512). Running this sequence through the model will result in indexing errors
 40%|████      | 405/1000 [13:47<13:49,  1.39s/it]  

In [3]:
# read  data
df = pd.read_parquet(f'/raid/deallab/SF_RAG_Data/ASQA/dev.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who is the original artist of sound of silence? 

qa_pairs :
[{'context': 'Sounds of Silence is the second studio album by Simon & Garfunkel, released on January 17, 1966. The album\'s title is a slight modification of the title of the duo\'s first major hit, "The Sound of Silence", which originally was released as "The Sounds of Silence". The song had earlier been released in an acoustic version on the album "Wednesday Morning, 3 A.M.", and later on the soundtrack to the movie "The Graduate". Without the knowledge of Paul Simon or Art Garfunkel, electric guitars, bass and drums were overdubbed by Columbia Records staff producer Tom Wilson on June 15, 1965. This new version was released as a single in September 1965, and opens the album.', 'question': 'Who is the original artist of sound of silence, the song, released in 1964?', 'short_answers': array(['Simon & Garfunkel', 'Paul Simon and Art Garfunkel',
        'Art Garfunkel', 'Paul Simon'], dtype=object), 'wikip

In [5]:
# create embedding document dataset.
evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'text'])
qa_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions', 'long_answers', 'short_answers'])

evidence_output_path = '/raid/deallab/SF_RAG_Data/ASQA/evidence_test.csv'
qa_output_path = '/raid/deallab/SF_RAG_Data/ASQA/qa_test.csv'
evidence_df.to_csv(evidence_output_path, index=False)
qa_df.to_csv(qa_output_path, index=False)

fetched_documents = []
for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
    try: 
        if idx == document_limit: break # change number of document to chunk/process
        sample_id = row['sample_id']
        evidences = row['wikipages']
        question = row['ambiguous_question']
        follow_up_question = [q["question"] for q in row['qa_pairs']]
        long_answers = [ann['long_answer'] for ann in row['annotations']]
        short_answers = [list(ann['short_answers']) for ann in row['qa_pairs']]
        for evidence in evidences:
            url = evidence['url']
            if url in fetched_documents: continue
            fetched_documents.append(url)
            page = requests.get(url)
            
            # Create a BeautifulSoup object
            soup = BeautifulSoup(page.text, 'html.parser')
            # get title
            title = soup.find(id='firstHeading').get_text()
            
            #extract content
            content = soup.find(class_='mw-content-ltr')
            parsed_doc = parse_document(content)
            
            # chunk document
            documents = [[]]
            
            for par in re.split(r'(?=\n#{1,4})', parsed_doc):
                tokenized_par = tokenizer.encode(par, add_special_tokens = False)
                length = len(tokenized_par)
                if len(documents[-1]) + length < TOK_LEN:
                    documents[-1].extend(tokenized_par)
                elif length > TOK_LEN:
                    begin = 0 
                    while begin < length:
                        if begin + TOK_LEN >= length:
                            documents.append(tokenized_par[begin:])
                            break
                        documents.append(tokenized_par[begin:begin + TOK_LEN])
                        begin += TOK_LEN - int(TOK_LEN * 0.1)
                else:
                    documents.append(tokenized_par)
                
            # print(len(documents))
            for doc in documents:
                doc_text = tokenizer.decode(doc)
                id = uuid4()
                #fetched_documents[url].append(id)
                evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, doc_text]
            
        qa_df.loc[len(qa_df)] = [uuid4(), sample_id, question, follow_up_question, long_answers, short_answers]
    except:
        print('continueing...')
    if (idx + 1 )% 100 == 0:
        print('Saving data...')
        evidence_df.to_csv(evidence_output_path, mode='a', header=False, index=False)
        qa_df.to_csv(qa_output_path, mode='a', header=False, index=False)
        evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'text'])
        qa_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions', 'long_answers', 'short_answers'])
        

evidence_df.to_csv(evidence_output_path, mode='a', header=False,index=False)
qa_df.to_csv(qa_output_path, mode='a', header=False, index=False)              
#print(fetched_documents)
print(evidence_df.head())
print(qa_df.head())

  0%|          | 1/948 [00:04<1:11:12,  4.51s/it]

0 1


  0%|          | 2/948 [00:06<46:13,  2.93s/it]  

1 2


  0%|          | 3/948 [00:09<45:18,  2.88s/it]

...

51 2


  6%|▌         | 53/948 [01:48<22:22,  1.50s/it]

52 3


  6%|▌         | 54/948 [01:49<19:29,  1.31s/it]

53 4
